<a href="https://colab.research.google.com/github/SajidAhmed-here/CD-Linter-with-yml/blob/main/Github_Smell_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import yaml
import re
import os
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np



def now_iso():
    return datetime.now(timezone.utc).isoformat()


def safe_str(val):
    return val if isinstance(val, str) else ""


def safe_dict(val):
    return val if isinstance(val, dict) else {}


def safe_jobs(wf: Dict) -> Dict:
    jobs = safe_dict(wf.get('jobs'))
    return {k: v for k, v in jobs.items() if isinstance(v, dict)}


def safe_yaml_load(raw: str) -> Optional[Dict]:

    if not isinstance(raw, str) or not raw.strip():
        return None
    # Replace GitHub expressions ONLY — do NOT touch '**'
    sanitised = re.sub(r"\$\{\{.*?\}\}", "GHA_EXPR_PLACEHOLDER", raw)
    try:
        data = yaml.safe_load(sanitised)
        return data if isinstance(data, dict) else None
    except Exception:
        return None


def extract_steps(job: Any) -> List[Dict]:
    if not isinstance(job, dict):
        return []
    steps = []
    raw_steps = job.get('steps', [])
    if isinstance(raw_steps, list):
        for s in raw_steps:
            if isinstance(s, dict):
                steps.append(s)
            elif isinstance(s, str):
                steps.append({'run': s})
    return steps


def find_line(raw: str, needle: str) -> int:
    for i, line in enumerate(raw.splitlines(), 1):
        if needle in line:
            return i
    return -1



class StateFlag:
    OPEN    = "OPEN"
    CLOSE   = "CLOSE"
    DEFAULT = "DEFAULT"

#
class GhaSmell:
    UNPINNED_ACTION           = "UnpinnedAction"
    BROAD_PERMISSIONS         = "BroadPermissions"
    MISSING_PERMISSIONS_BLOCK = "MissingPermissionsBlock"
    DANGEROUS_TOKEN           = "DangerousTokenUsage"
    INSECURE_SCRIPT           = "InsecureScript"
    PERMISSIVE_TRIGGER        = "PermissiveTrigger"
    SELF_HOSTED_RUNNER        = "SelfHostedRunner"
    PLAINTEXT_SECRET          = "PlaintextSecret"
    HARDCODED_ENV_VALUE       = "HardcodedEnvValue"
    PR_TARGET_RISK            = "PullRequestTargetRisk"
    UNPINNED_DOCKER_IMAGE     = "UnpinnedDockerImage"
    CONTINUE_ON_ERROR         = "ContinueOnError"
    ALLOW_FAILURE             = "AllowFailure"
    NO_CONCURRENCY            = "NoConcurrency"
    MISSING_TIMEOUT           = "MissingTimeout"
    RETRY_MISCONFIGURED       = "RetryMisconfigured"
    MISSING_CHECKOUT          = "MissingCheckout"
    MANUAL_ONLY_WORKFLOW      = "ManualOnlyWorkflow"
    MISSING_CACHE             = "MissingCache"
    REDUNDANT_STEPS           = "RedundantSteps"
    LARGE_MONOLITHIC_JOB      = "LargeMonolithicJob"


class GhaCategory:
    SECURITY      = "Security"
    RELIABILITY   = "Reliability"
    PERFORMANCE   = "Performance"


STAGE_ORDER = ['Test', 'Build', 'Deployment', 'Code Review',
               'Integration', 'Code Commit', 'Monitoring', 'Cross-Stage']

STAGE_COLORS = {
    'Test':        '#9999FF',
    'Build':       '#99CC99',
    'Deployment':  '#FFCC99',
    'Code Review': '#FF99FF',
    'Integration': '#99FFFF',
    'Code Commit': '#FFD700',
    'Monitoring':  '#FFB599',
    'Cross-Stage':       '#CCCCCC',
}


def infer_stage(job_name: str, step_name: str = '', uses: str = '') -> str:
    """
    Infer the pipeline stage from job/step name and action used.
    Matches the original Java get_stage() taxonomy.
    """
    text = (job_name + ' ' + step_name + ' ' + uses).lower()

    if any(w in text for w in [
        'test', 'junit', 'testng', 'spec', 'e2e', 'coverage',
        'cypress', 'playwright', 'pytest', 'rspec', 'jest',
        'surefire', 'verify', 'check',
    ]):
        return 'Test'

    if any(w in text for w in [
        'build', 'compile', 'setup', 'install', 'cache',
        'package', 'artifact', 'docker', 'image', 'make',
        'assemble', 'bundle', 'webpack', 'gradle', 'maven',
    ]):
        return 'Build'

    if any(w in text for w in [
        'deploy', 'release', 'publish', 'upload', 'push',
        'helm', 'kubectl', 'terraform', 'heroku', 'staging',
        'production', 'infra',
    ]):
        return 'Deployment'

    if any(w in text for w in [
        'review', 'lint', 'codeql', 'scan', 'analyze', 'analyse',
        'sonar', 'sast', 'dast', 'snyk', 'trivy', 'semgrep',
        'dependency', 'audit', 'eslint', 'pylint', 'rubocop',
    ]):
        return 'Code Review'

    if any(w in text for w in [
        'integration', 'merge', 'pr', 'pull_request', 'rebase',
    ]):
        return 'Integration'

    if any(w in text for w in [
        'commit', 'git', 'tag', 'version', 'changelog',
    ]):
        return 'Code Commit'

    if any(w in text for w in [
        'monitor', 'metrics', 'log', 'alert', 'report', 'notify',
        'notification', 'slack', 'email',
    ]):
        return 'Monitoring'

    return 'Cross-Stage'

SMELL_CATEGORY_COLORS = {
    GhaCategory.SECURITY:    '#E05C5C',
    GhaCategory.RELIABILITY: '#4A90D9',
    GhaCategory.PERFORMANCE: '#5CB85C',
}

# Known secrets env-var name patterns
SECRET_KEYWORDS = [
    'password', 'passwd', 'secret', 'token', 'apikey', 'api_key',
    'access_key', 'private_key', 'ssh_key', 'auth', 'credential',
    'aws_access_key_id', 'aws_secret_access_key',
]

# Dependency managers that benefit from caching
CACHE_TRIGGERS = [
    'npm install', 'yarn install', 'pnpm install',
    'pip install', 'pip3 install',
    'composer install',
    'bundle install',
    'gradle ', './gradlew ', 'mvn ', 'maven',
    'dotnet restore', 'nuget restore',
    'cargo build', 'go mod download',
]

BUILTIN_CACHE_ACTIONS = [
    'actions/setup-node',
    'actions/setup-python',
    'actions/setup-java',
    'actions/setup-go',
    'actions/setup-dotnet',
    'ruby/setup-ruby',
    'actions/cache',
    'gradle/gradle-build-action',
    'gradle/actions/setup-gradle',
]



class GitHubActionsSmellDetector:


    def __init__(self):
        self.issues: List[Dict] = []

    def analyze_workflow(self, workflow: Dict, project: str, raw: str) -> List[Dict]:
        issues = []
        issues += self._check_unpinned_actions(workflow, project, raw)
        issues += self._check_continue_on_error(workflow, project, raw)
        issues += self._check_missing_cache(workflow, project, raw)
        issues += self._check_permissions(workflow, project, raw)
        issues += self._check_no_concurrency(workflow, project, raw)
        issues += self._check_dangerous_token(workflow, project, raw)
        issues += self._check_insecure_script(workflow, project, raw)
        issues += self._check_permissive_trigger(workflow, project, raw)
        issues += self._check_self_hosted_runner(workflow, project, raw)
        issues += self._check_plaintext_secrets(workflow, project, raw)
        issues += self._check_pull_request_target(workflow, project, raw)
        issues += self._check_missing_timeout(workflow, project, raw)
        issues += self._check_missing_permissions_block(workflow, project, raw)
        issues += self._check_allow_failure(workflow, project, raw)
        issues += self._check_retry_misconfigured(workflow, project, raw)
        issues += self._check_missing_checkout(workflow, project, raw)
        issues += self._check_manual_only_workflow(workflow, project, raw)
        issues += self._check_hardcoded_env_value(workflow, project, raw)
        issues += self._check_unpinned_docker_image(workflow, project, raw)
        issues += self._check_redundant_steps(workflow, project, raw)
        issues += self._check_large_monolithic_job(workflow, project, raw)
        return issues


    def _issue(self, project: str, smell: str, category: str,
                entity: str, details: str, line: int,
                job_name: str = '', step_name: str = '', uses: str = '') -> Dict:
        return {
            'project':     project,
            'smell_type':  smell,
            'category':    category,
            'entity':      entity,
            'details':     details,
            'stage':       infer_stage(job_name, step_name, uses),
            'line_number': line,
            'timestamp':   now_iso(),
        }


    def _check_unpinned_actions(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            for step in extract_steps(job):
                uses = safe_str(step.get('uses')).strip()
                if not uses or '@' not in uses:
                    continue
                repo, ref = uses.split('@', 1)
                ref = ref.strip()
                if repo.startswith(('actions/', 'github/')):
                    continue
                if re.fullmatch(r'[0-9a-fA-F]{40}', ref):
                    continue
                step_name = safe_str(step.get('name'))
                issues.append(self._issue(
                    project, GhaSmell.UNPINNED_ACTION, GhaCategory.SECURITY,
                    entity=uses,
                    details=f"Third-party action pinned to '{ref}' instead of a full commit SHA: {uses}",
                    line=find_line(raw, uses),
                    job_name=job_name, step_name=step_name, uses=uses,
                ))
        return issues

    def _check_continue_on_error(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            for step in extract_steps(job):
                if step.get('continue-on-error') is True:
                    step_name = safe_str(step.get('name')) or 'unnamed step'
                    issues.append(self._issue(
                        project, GhaSmell.CONTINUE_ON_ERROR, GhaCategory.RELIABILITY,
                        entity=f"{job_name} > {step_name}",
                        details="continue-on-error: true hides failures and can mask broken steps",
                        line=find_line(raw, 'continue-on-error'),
                        job_name=job_name, step_name=step_name,
                    ))
        return issues

    def _check_missing_cache(self, wf, project, raw):
        raw_lower = raw.lower()
        needs_cache = any(t in raw_lower for t in CACHE_TRIGGERS)
        if not needs_cache:
            return []
        has_explicit_cache = 'actions/cache' in raw_lower
        has_builtin_cache  = any(a in raw_lower for a in BUILTIN_CACHE_ACTIONS)
        has_cache_input    = bool(re.search(
            r"cache\s*:\s*['\"]?(npm|pip|gradle|maven|bundler)['\"]?", raw, re.I))
        if has_explicit_cache or has_builtin_cache or has_cache_input:
            return []
        return [self._issue(
            project, GhaSmell.MISSING_CACHE, GhaCategory.PERFORMANCE,
            entity='workflow',
            details="Dependency install detected but no caching configured",
            line=-1,
            job_name='build', step_name='install',
        )]

    def _check_permissions(self, wf, project, raw):
        issues = []
        top_perms = wf.get('permissions')
        if top_perms == 'write-all' or (isinstance(top_perms, str) and 'write' in top_perms.lower()):
            issues.append(self._issue(
                project, GhaSmell.BROAD_PERMISSIONS, GhaCategory.SECURITY,
                entity='workflow',
                details="permissions: write-all grants maximum scope to all jobs",
                line=find_line(raw, 'write-all'),
                job_name='', step_name='permissions',
            ))
        for job_name, job in safe_jobs(wf).items():
            perms = job.get('permissions')
            if perms == 'write-all' or (isinstance(perms, str) and 'write' in perms.lower()):
                issues.append(self._issue(
                    project, GhaSmell.BROAD_PERMISSIONS, GhaCategory.SECURITY,
                    entity=job_name,
                    details=f"Job '{job_name}' uses write-all permissions",
                    line=find_line(raw, 'write-all'),
                    job_name=job_name, step_name='permissions',
                ))
        uses_github_token = 'GITHUB_TOKEN' in raw or 'github_token' in raw.lower()
        if 'permissions' not in raw and uses_github_token:
            issues.append(self._issue(
                project, GhaSmell.BROAD_PERMISSIONS, GhaCategory.SECURITY,
                entity='workflow',
                details="GITHUB_TOKEN used but no permissions block defined",
                line=-1,
                job_name='', step_name='permissions',
            ))
        return issues

    def _check_no_concurrency(self, wf, project, raw):
        if 'concurrency' in raw:
            return []
        triggers = wf.get('on', {})
        if isinstance(triggers, str):
            triggers = {triggers: None}
        if isinstance(triggers, list):
            triggers = {t: None for t in triggers}
        overlap_prone = {'push', 'pull_request', 'schedule', 'workflow_run'}
        if not any(t in overlap_prone for t in safe_dict(triggers)):
            return []
        return [self._issue(
            project, GhaSmell.NO_CONCURRENCY, GhaCategory.RELIABILITY,
            entity='workflow',
            details="No concurrency group — concurrent runs can cause race conditions",
            line=-1,
            job_name='integration', step_name='concurrency',
        )]

    def _check_dangerous_token(self, wf, project, raw):
        issues = []
        dangerous_ops = re.compile(
            r'\b(git\s+push|git\s+push\s+--force|gh\s+release|'
            r'gh\s+deploy|docker\s+push|helm\s+upgrade|kubectl\s+apply)\b', re.I)
        for job_name, job in safe_jobs(wf).items():
            for step in extract_steps(job):
                run = safe_str(step.get('run'))
                if not run:
                    continue
                if 'GITHUB_TOKEN' in run and dangerous_ops.search(run):
                    step_name = safe_str(step.get('name')) or 'unnamed step'
                    issues.append(self._issue(
                        project, GhaSmell.DANGEROUS_TOKEN, GhaCategory.SECURITY,
                        entity=f"{job_name} > {step_name}",
                        details="GITHUB_TOKEN used in a step that performs a push/deploy",
                        line=find_line(raw, 'GITHUB_TOKEN'),
                        job_name=job_name, step_name=step_name,
                    ))
        return issues

    def _check_insecure_script(self, wf, project, raw):
        issues = []
        pipe_pat = re.compile(
            r'(curl|wget)\b[^|]*\|\s*(sh|bash|zsh|dash|ksh|fish|pwsh)', re.I)
        for job_name, job in safe_jobs(wf).items():
            for step in extract_steps(job):
                run = safe_str(step.get('run'))
                m = pipe_pat.search(run)
                if m:
                    step_name = safe_str(step.get('name')) or 'unnamed step'
                    issues.append(self._issue(
                        project, GhaSmell.INSECURE_SCRIPT, GhaCategory.SECURITY,
                        entity=f"{job_name} > {step_name}",
                        details=f"Remote script executed via pipe: '{m.group(0).strip()}'",
                        line=find_line(raw, m.group(1)),
                        job_name=job_name, step_name=step_name,
                    ))
        return issues

    def _check_permissive_trigger(self, wf, project, raw):
        triggers = wf.get('on', {})
        if isinstance(triggers, str):
            triggers = {triggers: None}
        if isinstance(triggers, list):
            triggers = {t: None for t in triggers}
        if not isinstance(triggers, dict):
            return []
        issues = []
        for trigger in ('push', 'pull_request'):
            if trigger not in triggers:
                continue
            cfg = triggers[trigger]
            if isinstance(cfg, dict):
                if any(k in cfg for k in ('branches', 'branches-ignore', 'paths', 'paths-ignore')):
                    continue
            issues.append(self._issue(
                project, GhaSmell.PERMISSIVE_TRIGGER, GhaCategory.SECURITY,
                entity=f"on.{trigger}",
                details=f"Trigger '{trigger}' has no branch/path filter — runs on all branches",
                line=find_line(raw, trigger),
                job_name='integration', step_name=trigger,
            ))
        return issues

    def _check_self_hosted_runner(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            runs_on = job.get('runs-on', '')
            if isinstance(runs_on, list):
                runs_on_str = ' '.join(safe_str(x) for x in runs_on)
            else:
                runs_on_str = safe_str(runs_on)
            if 'self-hosted' in runs_on_str.lower():
                issues.append(self._issue(
                    project, GhaSmell.SELF_HOSTED_RUNNER, GhaCategory.SECURITY,
                    entity=job_name,
                    details=f"Job '{job_name}' uses a self-hosted runner",
                    line=find_line(raw, 'self-hosted'),
                    job_name=job_name, step_name='runner',
                ))
        return issues

    def _check_plaintext_secrets(self, wf, project, raw):
        issues = []
        def _scan_env(env_block, scope_label, job_name=''):
            if not isinstance(env_block, dict):
                return
            for k, v in env_block.items():
                k_low = k.lower().replace('-', '_')
                if not any(kw in k_low for kw in SECRET_KEYWORDS):
                    continue
                v_str = safe_str(v)
                if 'secrets.' not in v_str and 'GHA_EXPR_PLACEHOLDER' not in v_str:
                    issues.append(self._issue(
                        project, GhaSmell.PLAINTEXT_SECRET, GhaCategory.SECURITY,
                        entity=scope_label,
                        details=f"Env var '{k}' looks like a secret but is not from secrets context",
                        line=find_line(raw, k),
                        job_name=job_name, step_name='env',
                    ))
        _scan_env(wf.get('env'), 'workflow-level env')
        for job_name, job in safe_jobs(wf).items():
            _scan_env(safe_dict(job.get('env')), f"{job_name} env", job_name)
            for step in extract_steps(job):
                _scan_env(safe_dict(step.get('env')), f"{job_name} > step env", job_name)
        return issues

    def _check_pull_request_target(self, wf, project, raw):
        triggers = wf.get('on', {})
        if isinstance(triggers, str):
            triggers = {triggers: None}
        if isinstance(triggers, list):
            triggers = {t: None for t in triggers}
        if 'pull_request_target' not in safe_dict(triggers):
            return []
        for job_name, job in safe_jobs(wf).items():
            for step in extract_steps(job):
                uses = safe_str(step.get('uses'))
                with_block = safe_dict(step.get('with'))
                if 'actions/checkout' in uses:
                    ref = safe_str(with_block.get('ref', ''))
                    if 'head' in ref.lower() or 'pull_request' in ref.lower():
                        return [self._issue(
                            project, GhaSmell.PR_TARGET_RISK, GhaCategory.SECURITY,
                            entity=job_name,
                            details="pull_request_target + checkout of PR head enables script injection",
                            line=find_line(raw, 'pull_request_target'),
                            job_name=job_name, step_name='checkout',
                        )]
        return []

    def _check_missing_timeout(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            if 'timeout-minutes' in job:
                continue
            steps = extract_steps(job)
            if not any(s.get('run') or s.get('uses') for s in steps):
                continue
            issues.append(self._issue(
                project, GhaSmell.MISSING_TIMEOUT, GhaCategory.RELIABILITY,
                entity=job_name,
                details=f"Job '{job_name}' has no timeout-minutes — runaway jobs consume CI quota",
                line=find_line(raw, job_name + ':'),
                job_name=job_name,
            ))
        return issues


    def _check_missing_permissions_block(self, wf, project, raw):
        # If there IS a top-level permissions key, this smell doesn't apply
        if 'permissions' in wf:
            return []
        # Only flag when GITHUB_TOKEN is actually in play
        if 'GITHUB_TOKEN' not in raw and 'github_token' not in raw.lower():
            return []
        return [self._issue(
            project, GhaSmell.MISSING_PERMISSIONS_BLOCK, GhaCategory.SECURITY,
            entity='workflow',
            details="No top-level permissions block defined — GitHub defaults may grant write access on all scopes",
            line=-1,
            job_name='', step_name='permissions',
        )]


    def _check_allow_failure(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            if job.get('continue-on-error') is True:
                issues.append(self._issue(
                    project, GhaSmell.ALLOW_FAILURE, GhaCategory.RELIABILITY,
                    entity=job_name,
                    details=f"Job '{job_name}' has continue-on-error: true — job failures are silently ignored",
                    line=find_line(raw, 'continue-on-error'),
                    job_name=job_name,
                ))
        return issues


    RETRY_ACTIONS = re.compile(
        r'nick-fields/retry|Wandalen/wretry\.action|EndBug/add-and-commit.*retry',
        re.I
    )

    def _check_retry_misconfigured(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            # strategy.max-parallel: 1 on a matrix defeats the whole point
            strategy = safe_dict(job.get('strategy'))
            if isinstance(strategy.get('matrix'), dict) and strategy.get('max-parallel') == 1:
                issues.append(self._issue(
                    project, GhaSmell.RETRY_MISCONFIGURED, GhaCategory.RELIABILITY,
                    entity=job_name,
                    details=f"Job '{job_name}' uses a matrix but sets max-parallel: 1, serialising all runs",
                    line=find_line(raw, 'max-parallel'),
                    job_name=job_name,
                ))

            # Retry action used without a timeout or condition parameter
            for step in extract_steps(job):
                uses = safe_str(step.get('uses'))
                if self.RETRY_ACTIONS.search(uses):
                    with_block = safe_dict(step.get('with'))
                    if 'timeout_minutes' not in with_block and 'timeout' not in with_block:
                        step_name = safe_str(step.get('name')) or uses
                        issues.append(self._issue(
                            project, GhaSmell.RETRY_MISCONFIGURED, GhaCategory.RELIABILITY,
                            entity=f"{job_name} > {step_name}",
                            details=f"Retry action '{uses}' used without a timeout — could retry indefinitely",
                            line=find_line(raw, uses),
                            job_name=job_name, step_name=step_name, uses=uses,
                        ))
        return issues


    SOURCE_CODE_PATTERNS = re.compile(
        r'\b(npm|yarn|pnpm|pip|pytest|gradle|mvn|make|go\s+build|cargo|'
        r'eslint|pylint|rubocop|tsc|jest|rspec|dotnet\s+build)\b',
        re.I
    )

    def _check_missing_checkout(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            steps = extract_steps(job)
            has_checkout = any('actions/checkout' in safe_str(s.get('uses')) for s in steps)
            if has_checkout:
                continue
            # Only flag if the job actually runs source-code operations
            all_run = ' '.join(safe_str(s.get('run')) for s in steps)
            if self.SOURCE_CODE_PATTERNS.search(all_run):
                issues.append(self._issue(
                    project, GhaSmell.MISSING_CHECKOUT, GhaCategory.RELIABILITY,
                    entity=job_name,
                    details=f"Job '{job_name}' runs source-code commands but has no actions/checkout step",
                    line=find_line(raw, job_name + ':'),
                    job_name=job_name,
                ))
        return issues


    def _check_manual_only_workflow(self, wf, project, raw):
        triggers = wf.get('on', {})
        if isinstance(triggers, str):
            triggers = {triggers: None}
        if isinstance(triggers, list):
            triggers = {t: None for t in triggers}
        if not isinstance(triggers, dict):
            return []

        automated = {'push', 'pull_request', 'pull_request_target',
                     'schedule', 'workflow_run', 'release', 'create', 'delete'}
        has_automation = any(t in automated for t in triggers)
        has_manual     = 'workflow_dispatch' in triggers

        if has_manual and not has_automation:
            return [self._issue(
                project, GhaSmell.MANUAL_ONLY_WORKFLOW, GhaCategory.RELIABILITY,
                entity='workflow',
                details="Workflow is triggered only by workflow_dispatch — no automated CI trigger defined",
                line=find_line(raw, 'workflow_dispatch'),
                job_name='integration', step_name='trigger',
            )]
        return []


    CONFIG_ENV_PATTERNS = re.compile(
        r'\b(environment|env|region|zone|cluster|namespace|registry|'
        r'image_name|app_name|service_name|stage|target|profile|'
        r'aws_region|azure_location|gcp_project|docker_registry)\b',
        re.I
    )
    HARDCODED_VALUE = re.compile(r'^[A-Za-z][A-Za-z0-9_\-\.:/]{3,}$')

    def _check_hardcoded_env_value(self, wf, project, raw):
        issues = []

        def _scan(env_block, scope, job_name=''):
            if not isinstance(env_block, dict):
                return
            for k, v in env_block.items():
                if not self.CONFIG_ENV_PATTERNS.search(k):
                    continue
                v_str = safe_str(v)
                if not v_str:
                    continue
                # Skip expressions and anything from secrets/vars context
                if 'GHA_EXPR_PLACEHOLDER' in v_str or '${{' in v_str:
                    continue
                if v_str.startswith('${{') or 'vars.' in v_str or 'secrets.' in v_str:
                    continue
                if self.HARDCODED_VALUE.match(v_str):
                    issues.append(self._issue(
                        project, GhaSmell.HARDCODED_ENV_VALUE, GhaCategory.SECURITY,
                        entity=scope,
                        details=f"Env var '{k}' uses hardcoded value '{v_str}' — use ${{{{ vars.{k.upper()} }}}} instead",
                        line=find_line(raw, k),
                        job_name=job_name, step_name='env',
                    ))

        _scan(wf.get('env'), 'workflow-level env')
        for job_name, job in safe_jobs(wf).items():
            _scan(safe_dict(job.get('env')), f"{job_name} env", job_name)
            for step in extract_steps(job):
                _scan(safe_dict(step.get('env')), f"{job_name} > step env", job_name)
        return issues


    def _check_unpinned_docker_image(self, wf, project, raw):
        issues = []
        for job_name, job in safe_jobs(wf).items():
            container = job.get('container')
            if not container:
                continue
            image = safe_str(container if isinstance(container, str)
                             else safe_dict(container).get('image', ''))
            if not image:
                continue
            # Safe: image pinned to a digest  e.g. ubuntu@sha256:abc123...
            if '@sha256:' in image:
                continue
            # Smell: :latest tag or no tag at all (implicitly latest)
            tag = image.split(':')[-1] if ':' in image else 'latest'
            if tag == 'latest' or not re.match(r'^[0-9a-fA-F]{7,}$', tag):
                issues.append(self._issue(
                    project, GhaSmell.UNPINNED_DOCKER_IMAGE, GhaCategory.SECURITY,
                    entity=job_name,
                    details=f"Container image '{image}' is not pinned to a digest — use image@sha256:<digest>",
                    line=find_line(raw, image),
                    job_name=job_name, step_name='container',
                ))
        return issues


    REDUNDANT_STEP_THRESHOLD = 3

    def _check_redundant_steps(self, wf, project, raw):
        from collections import Counter
        step_job_count: Counter = Counter()

        for job_name, job in safe_jobs(wf).items():
            seen_in_job = set()
            for step in extract_steps(job):
                key = safe_str(step.get('uses')) or safe_str(step.get('run', ''))[:80]
                key = key.strip()
                if key and key not in seen_in_job:
                    step_job_count[key] += 1
                    seen_in_job.add(key)

        issues = []
        for step_key, count in step_job_count.items():
            if count >= self.REDUNDANT_STEP_THRESHOLD:
                issues.append(self._issue(
                    project, GhaSmell.REDUNDANT_STEPS, GhaCategory.PERFORMANCE,
                    entity='multiple jobs',
                    details=f"Step '{step_key[:60]}' is duplicated across {count} jobs — consider a reusable workflow",
                    line=find_line(raw, step_key[:40]),
                    job_name='build', step_name='duplicate',
                ))
        return issues


    MONOLITHIC_JOB_STEP_THRESHOLD = 10

    def _check_large_monolithic_job(self, wf, project, raw):
        jobs = safe_jobs(wf)
        if len(jobs) != 1:          # parallelism is already present
            return []
        job_name, job = next(iter(jobs.items()))
        steps = extract_steps(job)
        if len(steps) >= self.MONOLITHIC_JOB_STEP_THRESHOLD:
            return [self._issue(
                project, GhaSmell.LARGE_MONOLITHIC_JOB, GhaCategory.PERFORMANCE,
                entity=job_name,
                details=(f"Workflow has a single job '{job_name}' with {len(steps)} steps "
                         f"and no parallelism — consider splitting into parallel jobs"),
                line=find_line(raw, job_name + ':'),
                job_name=job_name,
            )]
        return []




def _smell_color(smell_name: str) -> str:
    cat = SMELL_TO_CATEGORY.get(smell_name, GhaCategory.PERFORMANCE)
    return SMELL_CATEGORY_COLORS[cat]


def draw_bar_chart(counts: pd.Series, title: str, path: str,
                   is_stage_chart: bool = False):
    if counts.empty:
        print(f"  ⚠️  No data to chart: {title}")
        return

    labels = counts.index.tolist()
    values = counts.values.tolist()
    colors = (
        [STAGE_COLORS.get(s, '#CCCCCC') for s in labels]
        if is_stage_chart
        else [_smell_color(s) for s in labels]
    )

    fig_h = max(5, len(labels) * 0.55 + 2.5)
    fig, ax = plt.subplots(figsize=(13, fig_h))
    fig.patch.set_facecolor('#F7F9FC')
    ax.set_facecolor('#F7F9FC')

    y = np.arange(len(labels))
    bars = ax.barh(y, values, color=colors, edgecolor='white',
                   linewidth=0.8, height=0.65)

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.012,
                bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left',
                fontsize=9, fontweight='bold', color='#333333')

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9.5)
    ax.invert_yaxis()
    ax.set_xlabel('Occurrence Count', fontsize=11, labelpad=8)
    ax.set_xlim(0, max(values) * 1.15)
    ax.xaxis.grid(True, linestyle='--', alpha=0.45, color='#CCCCCC', zorder=0)
    ax.set_axisbelow(True)
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color('#CCCCCC')
    ax.set_title(title, fontsize=13, fontweight='bold', color='#1A1A2E',
                 pad=12, loc='left')

    if not is_stage_chart:
        patches = [
            mpatches.Patch(color=SMELL_CATEGORY_COLORS[GhaCategory.SECURITY],    label='Security'),
            mpatches.Patch(color=SMELL_CATEGORY_COLORS[GhaCategory.RELIABILITY], label='Reliability'),
            mpatches.Patch(color=SMELL_CATEGORY_COLORS[GhaCategory.PERFORMANCE], label='Performance'),
        ]
        ax.legend(handles=patches, loc='lower right', fontsize=9,
                  framealpha=0.85, edgecolor='#CCCCCC')

    plt.tight_layout(pad=1.5)
    plt.savefig(path, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"  📊 Chart saved → {path}")



def _analyze_column(df: pd.DataFrame, yaml_col: str, project_col: str) -> pd.DataFrame:
    detector = GitHubActionsSmellDetector()
    all_issues = []

    for idx, row in df.iterrows():
        raw = row.get(yaml_col)
        if not isinstance(raw, str) or not raw.strip():
            continue
        project = safe_str(row.get(project_col)) or f"Project_{idx + 1}"
        wf = safe_yaml_load(raw)
        if not isinstance(wf, dict):
            continue
        all_issues.extend(detector.analyze_workflow(wf, project, raw.strip()))

    return pd.DataFrame(all_issues) if all_issues else pd.DataFrame()



def analyze_and_compare_columns(
    input_csv: str,
    project_column: str        = "URL",
    ground_truth_column: str   = "Code",
    llm_output_column: str     = "LLM_YML_Output",
    output_dir: str            = "./gha_results",
):
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*72}")
    print("🔍 GitHub Actions Smell Detector — Dual-Column Comparison")
    print(f"{'='*72}\n")

    if not os.path.exists(input_csv):
        print(f"❌ File not found: {input_csv}")
        return

    df = pd.read_csv(input_csv)
    print(f"✓ Loaded {len(df)} rows from {input_csv}")

    results = {}
    for label, col in [("Ground Truth", ground_truth_column),
                        ("LLM Output",   llm_output_column)]:
        if col not in df.columns:
            print(f"⚠️  Column '{col}' not found — skipping.")
            results[label] = pd.DataFrame()
            continue

        print(f"\n📋 Analysing [{label}] → column '{col}' ...")
        issue_df = _analyze_column(df, col, project_column)

        if issue_df.empty:
            print(f"  ⚠️  No smells detected in '{col}'.")
        else:
            csv_path = os.path.join(output_dir, f"smells_{label.lower().replace(' ','_')}.csv")
            issue_df.to_csv(csv_path, index=False)
            counts = issue_df['smell_type'].value_counts()
            print(f"  ✅ {len(issue_df)} smells ({counts.nunique()} types) saved → {csv_path}")

        results[label] = issue_df

    # ── Bar Charts ────────────────────────────────────────────────────────
    print("\n🎨 Generating charts ...")
    for label, issue_df in results.items():
        if issue_df.empty:
            continue
        slug = label.lower().replace(' ', '_')

        draw_bar_chart(
            issue_df['smell_type'].value_counts().sort_values(ascending=False),
            title=f"GHA Smells by Type — {label}",
            path=os.path.join(output_dir, f"chart_smells_{slug}.png"),
        )
        draw_bar_chart(
            issue_df['stage'].value_counts().sort_values(ascending=False),
            title=f"GHA Smells by Stage — {label}",
            path=os.path.join(output_dir, f"chart_stages_{slug}.png"),
            is_stage_chart=True,
        )

    # ── Side-by-side summary table ────────────────────────────────────────
    gt_c   = results["Ground Truth"]['smell_type'].value_counts() \
             if not results["Ground Truth"].empty else pd.Series(dtype=int)
    llm_c  = results["LLM Output"]['smell_type'].value_counts() \
             if not results["LLM Output"].empty else pd.Series(dtype=int)

    print(f"\n{'='*72}")
    print("📊 SMELL TYPE COMPARISON")
    print(f"{'='*72}")
    print(f"  {'Smell Type':<38} {'Ground Truth':>14}  {'LLM Output':>12}")
    print(f"  {'-'*66}")
    for smell in sorted(set(gt_c.index) | set(llm_c.index)):
        print(f"  {smell:<38} {gt_c.get(smell, 0):>14}  {llm_c.get(smell, 0):>12}")
    print(f"  {'─'*66}")
    print(f"  {'TOTAL':<38} {int(gt_c.sum()):>14}  {int(llm_c.sum()):>12}")
    print(f"{'='*72}")

    # ── Stage-wise cross-tab ──────────────────────────────────────────────
    for label, issue_df in results.items():
        if issue_df.empty:
            continue
        print(f"\n📋 Stage × Smell cross-tab — {label}")
        ct = pd.crosstab(issue_df['stage'], issue_df['smell_type'], margins=True)
        ct = ct.reindex(index=STAGE_ORDER + ['All'], fill_value=0).fillna(0).astype(int)
        print(ct.to_string())

    print(f"\n✅ All outputs saved in: {output_dir}")
    print(f"{'='*72}\n")



SMELL_TO_CATEGORY = {
    # Security
    GhaSmell.UNPINNED_ACTION:           GhaCategory.SECURITY,
    GhaSmell.BROAD_PERMISSIONS:         GhaCategory.SECURITY,
    GhaSmell.MISSING_PERMISSIONS_BLOCK: GhaCategory.SECURITY,
    GhaSmell.DANGEROUS_TOKEN:           GhaCategory.SECURITY,
    GhaSmell.INSECURE_SCRIPT:           GhaCategory.SECURITY,
    GhaSmell.PERMISSIVE_TRIGGER:        GhaCategory.SECURITY,
    GhaSmell.SELF_HOSTED_RUNNER:        GhaCategory.SECURITY,
    GhaSmell.PLAINTEXT_SECRET:          GhaCategory.SECURITY,
    GhaSmell.HARDCODED_ENV_VALUE:       GhaCategory.SECURITY,
    GhaSmell.PR_TARGET_RISK:            GhaCategory.SECURITY,
    GhaSmell.UNPINNED_DOCKER_IMAGE:     GhaCategory.SECURITY,
    # Reliability
    GhaSmell.CONTINUE_ON_ERROR:         GhaCategory.RELIABILITY,
    GhaSmell.ALLOW_FAILURE:             GhaCategory.RELIABILITY,
    GhaSmell.NO_CONCURRENCY:            GhaCategory.RELIABILITY,
    GhaSmell.MISSING_TIMEOUT:           GhaCategory.RELIABILITY,
    GhaSmell.RETRY_MISCONFIGURED:       GhaCategory.RELIABILITY,
    GhaSmell.MISSING_CHECKOUT:          GhaCategory.RELIABILITY,
    GhaSmell.MANUAL_ONLY_WORKFLOW:      GhaCategory.RELIABILITY,
    # Performance
    GhaSmell.MISSING_CACHE:             GhaCategory.PERFORMANCE,
    GhaSmell.REDUNDANT_STEPS:           GhaCategory.PERFORMANCE,
    GhaSmell.LARGE_MONOLITHIC_JOB:      GhaCategory.PERFORMANCE,
}


def statistical_summary(
    input_csv: str,
    project_column: str      = "URL",
    ground_truth_column: str = "Code",
    llm_output_column: str   = "LLM_YML_Output",
):

    print(f"\n{'='*82}")
    print("📈 STATISTICAL SUMMARY  —  LOC-Normalised Smell Density")
    print(f"    density = smells / total lines of code  (standard research metric)")
    print(f"{'='*82}")

    df = pd.read_csv(input_csv)

    # ── Compute LOC directly from the raw YAML columns ─────────────────────
    gt_loc  = int(df[ground_truth_column].fillna("").astype(str)
                  .apply(lambda x: len(x.splitlines())).sum())
    llm_loc = int(df[llm_output_column].fillna("").astype(str)
                  .apply(lambda x: len(x.splitlines())).sum())

    n_gt_wf  = int(df[ground_truth_column].notna().sum())
    n_llm_wf = int(df[llm_output_column].notna().sum())

    gt_df  = _analyze_column(df, ground_truth_column, project_column)
    llm_df = _analyze_column(df, llm_output_column,   project_column)

    gt_total  = len(gt_df)
    llm_total = len(llm_df)

    def density(count, loc):
        return (count / loc * 1000) if loc else 0.0

    gt_d  = density(gt_total,  gt_loc)
    llm_d = density(llm_total, llm_loc)

    W = 82  # table width

    print(f"\n{'─'*W}")
    print("  1.  LOC BASELINE")
    print(f"{'─'*W}")
    print(f"  {'Metric':<35} {'Ground Truth':>18}  {'LLM Output':>16}")
    print(f"  {'-'*72}")
    print(f"  {'Workflows analysed':<35} {n_gt_wf:>18,}  {n_llm_wf:>16,}")
    print(f"  {'Total lines of code (LOC)':<35} {gt_loc:>18,}  {llm_loc:>16,}")
    avg_gt  = gt_loc  / n_gt_wf  if n_gt_wf  else 0
    avg_llm = llm_loc / n_llm_wf if n_llm_wf else 0
    print(f"  {'Avg LOC per workflow':<35} {avg_gt:>18.1f}  {avg_llm:>16.1f}")
    print(f"  {'Total smells detected':<35} {gt_total:>18,}  {llm_total:>16,}")

    print(f"\n{'─'*W}")
    print("  2.  OVERALL SMELL DENSITY  (smells per 1,000 LOC)")
    print(f"{'─'*W}")
    print(f"  {'Metric':<35} {'Ground Truth':>18}  {'LLM Output':>16}")
    print(f"  {'-'*72}")
    print(f"  {'Total smells':<35} {gt_total:>18,}  {llm_total:>16,}")
    print(f"  {'Total LOC':<35} {gt_loc:>18,}  {llm_loc:>16,}")
    print(f"  {'Smell density (per KLOC)':<35} {gt_d:>18.4f}  {llm_d:>16.4f}")

    print(f"\n{'─'*W}")
    print("  3.  SMELL DENSITY BY CATEGORY  (smells per 1,000 LOC)")
    print(f"{'─'*W}")
    print(f"  {'Category':<20} {'GT Count':>10}  {'GT /KLOC':>10}  "
          f"{'LLM Count':>10}  {'LLM /KLOC':>10}")
    print(f"  {'-'*64}")

    for cat in [GhaCategory.SECURITY, GhaCategory.RELIABILITY, GhaCategory.PERFORMANCE]:
        smells_in_cat = {s for s, c in SMELL_TO_CATEGORY.items() if c == cat}
        gt_c  = int(gt_df[gt_df['smell_type'].isin(smells_in_cat)].shape[0])   if not gt_df.empty  else 0
        llm_c = int(llm_df[llm_df['smell_type'].isin(smells_in_cat)].shape[0]) if not llm_df.empty else 0
        print(f"  {cat:<20} {gt_c:>10,}  {density(gt_c, gt_loc):>10.4f}  "
              f"{llm_c:>10,}  {density(llm_c, llm_loc):>10.4f}")

    print(f"  {'─'*64}")
    print(f"  {'TOTAL':<20} {gt_total:>10,}  {gt_d:>10.4f}  "
          f"{llm_total:>10,}  {llm_d:>10.4f}")

    print(f"\n{'─'*W}")
    print("  4.  SMELL DENSITY BY PIPELINE STAGE  (smells per 1,000 LOC)")
    print(f"{'─'*W}")
    print(f"  {'Stage':<20} {'GT Count':>10}  {'GT /KLOC':>10}  "
          f"{'LLM Count':>10}  {'LLM /KLOC':>10}")
    print(f"  {'-'*64}")

    gt_stage  = gt_df['stage'].value_counts()  if not gt_df.empty  else pd.Series(dtype=int)
    llm_stage = llm_df['stage'].value_counts() if not llm_df.empty else pd.Series(dtype=int)

    for stage in STAGE_ORDER:
        gt_c  = int(gt_stage.get(stage,  0))
        llm_c = int(llm_stage.get(stage, 0))
        print(f"  {stage:<20} {gt_c:>10,}  {density(gt_c, gt_loc):>10.4f}  "
              f"{llm_c:>10,}  {density(llm_c, llm_loc):>10.4f}")

    print(f"  {'─'*64}")
    print(f"  {'TOTAL':<20} {gt_total:>10,}  {gt_d:>10.4f}  "
          f"{llm_total:>10,}  {llm_d:>10.4f}")

    print(f"\n{'─'*W}")
    print("  5.  SMELL DENSITY BY TYPE  (smells per 1,000 LOC)")
    print(f"{'─'*W}")
    print(f"  {'Smell Type':<30} {'Category':<13} {'GT Cnt':>7}  {'GT /KLOC':>9}  "
          f"{'LLM Cnt':>8}  {'LLM /KLOC':>10}")
    print(f"  {'-'*82}")

    gt_type  = gt_df['smell_type'].value_counts()  if not gt_df.empty  else pd.Series(dtype=int)
    llm_type = llm_df['smell_type'].value_counts() if not llm_df.empty else pd.Series(dtype=int)
    all_types = sorted(set(list(gt_type.index) + list(llm_type.index)))

    for smell in all_types:
        cat   = SMELL_TO_CATEGORY.get(smell, '?')
        gt_c  = int(gt_type.get(smell,  0))
        llm_c = int(llm_type.get(smell, 0))
        print(f"  {smell:<30} {cat:<13} {gt_c:>7,}  {density(gt_c, gt_loc):>9.4f}  "
              f"{llm_c:>8,}  {density(llm_c, llm_loc):>10.4f}")

    print(f"  {'─'*82}")
    print(f"  {'TOTAL':<44} {gt_total:>7,}  {gt_d:>9.4f}  "
          f"{llm_total:>8,}  {llm_d:>10.4f}")
    print(f"\n{'='*82}\n")



from scipy.stats import mannwhitneyu   # add this import near the top of your file


def mann_whitney_summary(
    input_csv: str,
    project_column: str      = "URL",
    ground_truth_column: str = "Code",
    llm_output_column: str   = "LLM_YML_Output",
):
    """
    Per-workflow smell-density Mann-Whitney U test (two-sided) with
    rank-biserial correlation effect size.

    For every grouping (overall / by category / by stage / by smell type) we
    build two vectors:
        gt_densities[i]  = smells_in_group / loc_of_workflow_i  * 1000
        llm_densities[i] = smells_in_group / loc_of_workflow_i  * 1000
    and run scipy.stats.mannwhitneyu (two-sided, exact=False).

    Effect size — rank-biserial correlation r:
        r = 1 - (2 * U) / (n1 * n2)
        |r| < 0.1  → negligible
        |r| < 0.3  → small
        |r| < 0.5  → medium
              else → large

    Note: most per-workflow vectors are zero-inflated (median = 0),
    so the test detects distributional differences in the upper tail.
    The rank-biserial r provides the effect magnitude independent of n.
    Rows where BOTH vectors are all-zero are marked '—' (test not meaningful).
    """
    import numpy as np
    from scipy.stats import mannwhitneyu

    W = 104

    def effect_label(r: float) -> str:
        a = abs(r)
        if a < 0.1:  return "negligible"
        if a < 0.3:  return "small"
        if a < 0.5:  return "medium"
        return "large"

    print(f"\n{'='*W}")
    print("🧪 MANN-WHITNEY U TEST  —  Per-Workflow Smell Density  (Ground Truth vs LLM)")
    print(f"    H₀: distributions of smell density are identical")
    print(f"    density = smells / LOC × 1,000  |  two-sided  |  effect size = rank-biserial r")
    print(f"    Note: zero-inflated data — test detects upper-tail distributional differences")
    print(f"{'='*W}")

    df = pd.read_csv(input_csv)

    # ── Per-workflow LOC ──────────────────────────────────────────────────────
    def wf_loc(series: pd.Series) -> pd.Series:
        return series.fillna("").astype(str).apply(lambda x: max(len(x.splitlines()), 1))

    gt_loc_vec  = wf_loc(df[ground_truth_column])
    llm_loc_vec = wf_loc(df[llm_output_column])

    # ── Detect smells per workflow ────────────────────────────────────────────
    def per_wf_issues(yaml_col: str, loc_vec: pd.Series):
        detector = GitHubActionsSmellDetector()
        records = []
        for idx, row in df.iterrows():
            raw = row.get(yaml_col)
            if not isinstance(raw, str) or not raw.strip():
                continue
            project = safe_str(row.get(project_column)) or f"Project_{idx+1}"
            wf = safe_yaml_load(raw)
            if not isinstance(wf, dict):
                continue
            issues = detector.analyze_workflow(wf, project, raw.strip())
            loc = int(loc_vec.iloc[idx])
            for iss in issues:
                records.append((idx, loc, iss))
        return records

    gt_records  = per_wf_issues(ground_truth_column, gt_loc_vec)
    llm_records = per_wf_issues(llm_output_column,   llm_loc_vec)

    n_gt_wf  = int(df[ground_truth_column].notna().sum())
    n_llm_wf = int(df[llm_output_column].notna().sum())

    # ── Helper: build density vectors for a given filter ─────────────────────
    def density_vectors(gt_recs, llm_recs,
                        gt_loc_vec, llm_loc_vec,
                        n_gt, n_llm,
                        filter_fn=None):
        gt_counts  = np.zeros(n_gt,  dtype=float)
        llm_counts = np.zeros(n_llm, dtype=float)

        for idx, loc, iss in gt_recs:
            if filter_fn is None or filter_fn(iss):
                gt_counts[idx] += 1

        for idx, loc, iss in llm_recs:
            if filter_fn is None or filter_fn(iss):
                llm_counts[idx] += 1

        gt_vec = np.array([
            gt_counts[i] / gt_loc_vec.iloc[i] * 1000
            for i in range(n_gt)
        ])
        llm_vec = np.array([
            llm_counts[i] / llm_loc_vec.iloc[i] * 1000
            for i in range(n_llm)
        ])
        return gt_vec, llm_vec

    # ── Run the test, compute effect size, format one row ────────────────────
    def mw_row(label: str, col_label: str, filter_fn=None):
        gt_vec, llm_vec = density_vectors(
            gt_records, llm_records,
            gt_loc_vec, llm_loc_vec,
            n_gt_wf, n_llm_wf,
            filter_fn=filter_fn,
        )

        # Skip rows where both vectors are all-zero (test is not meaningful)
        both_zero = (gt_vec.sum() == 0) and (llm_vec.sum() == 0)

        try:
            stat, p = mannwhitneyu(gt_vec, llm_vec, alternative='two-sided')
            n1, n2  = len(gt_vec), len(llm_vec)
            # rank-biserial r: +1 = LLM always > GT, -1 = GT always > LLM
            r = 1 - (2 * stat) / (n1 * n2)
        except ValueError:
            stat, p, r = float('nan'), float('nan'), float('nan')

        if both_zero:
            sig = "—"
            r   = float('nan')
            eff = "—"
        else:
            sig = ("***" if p < 0.001 else
                   "**"  if p < 0.01  else
                   "*"   if p < 0.05  else
                   "ns")
            eff = effect_label(r)

        gt_med  = float(np.median(gt_vec))
        llm_med = float(np.median(llm_vec))
        r_str   = f"{r:+.3f}" if not (both_zero or r != r) else "   n/a"

        return (label, col_label,
                f"{gt_med:.4f}", f"{llm_med:.4f}",
                f"{stat:.1f}", f"{p:.4f}", sig,
                r_str, eff)

    # ── Print a formatted table ───────────────────────────────────────────────
    def print_section(title: str, rows: list):
        print(f"\n{'─'*W}")
        print(f"  {title}")
        print(f"{'─'*W}")
        hdr = (f"  {'Grouping':<26} {'Sub-group':<22} {'GT Med/KLOC':>11}"
               f"  {'LLM Med/KLOC':>12}  {'U stat':>10}  {'p-value':>8}"
               f"  {'Sig':>3}  {'r':>6}  {'Effect':<10}")
        print(hdr)
        print(f"  {'-'*(W-2)}")
        for row in rows:
            print(f"  {row[0]:<26} {row[1]:<22} {row[2]:>11}"
                  f"  {row[3]:>12}  {row[4]:>10}  {row[5]:>8}"
                  f"  {row[6]:>3}  {row[7]:>6}  {row[8]:<10}")
        print(f"\n  Significance : *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")
        print(f"  Effect size  : |r| < 0.1 negligible | < 0.3 small | < 0.5 medium | >= 0.5 large")
        print(f"  '—' rows     : both GT and LLM vectors are all-zero -> test not meaningful")

    # ── 1. Overall ────────────────────────────────────────────────────────────
    print_section("1.  OVERALL", [mw_row("All smells", "—")])

    # ── 2. By category ───────────────────────────────────────────────────────
    cat_rows = []
    for cat in [GhaCategory.SECURITY, GhaCategory.RELIABILITY, GhaCategory.PERFORMANCE]:
        smells_in = {s for s, c in SMELL_TO_CATEGORY.items() if c == cat}
        cat_rows.append(mw_row("Category", cat,
                               filter_fn=lambda iss, si=smells_in: iss['smell_type'] in si))
    print_section("2.  BY CATEGORY", cat_rows)

    # ── 3. By pipeline stage ──────────────────────────────────────────────────
    stage_rows = [
        mw_row("Stage", stage,
               filter_fn=lambda iss, s=stage: iss['stage'] == s)
        for stage in STAGE_ORDER
    ]
    print_section("3.  BY PIPELINE STAGE", stage_rows)

    # ── 4. By smell type ──────────────────────────────────────────────────────
    all_smell_types = sorted(SMELL_TO_CATEGORY.keys())
    smell_rows = [
        mw_row(SMELL_TO_CATEGORY.get(smell, '?'), smell,
               filter_fn=lambda iss, s=smell: iss['smell_type'] == s)
        for smell in all_smell_types
    ]
    print_section("4.  BY SMELL TYPE", smell_rows)

    print(f"\n{'='*W}\n")



if __name__ == "__main__":
    INPUT_CSV      = "/content/GitHubActions_llama3.1-8b_output.csv"
    PROJECT_COLUMN = "URL"
    GT_COLUMN      = "Code"
    LLM_COLUMN     = "LLM_YML_Output"
    OUTPUT_DIR     = "./gha_results"

    analyze_and_compare_columns(
        input_csv=INPUT_CSV,
        project_column=PROJECT_COLUMN,
        ground_truth_column=GT_COLUMN,
        llm_output_column=LLM_COLUMN,
        output_dir=OUTPUT_DIR,
    )
    mann_whitney_summary(
         input_csv=INPUT_CSV,
         project_column=PROJECT_COLUMN,
         ground_truth_column=GT_COLUMN,
         llm_output_column=LLM_COLUMN,
     )

    statistical_summary(
        input_csv=INPUT_CSV,
        project_column=PROJECT_COLUMN,
        ground_truth_column=GT_COLUMN,
        llm_output_column=LLM_COLUMN,
    )


🔍 GitHub Actions Smell Detector — Dual-Column Comparison

✓ Loaded 1318 rows from /content/GitHubActions_llama3.1-8b_output.csv

📋 Analysing [Ground Truth] → column 'Code' ...
  ✅ 616 smells (11 types) saved → ./gha_results/smells_ground_truth.csv

📋 Analysing [LLM Output] → column 'LLM_YML_Output' ...
  ✅ 1140 smells (11 types) saved → ./gha_results/smells_llm_output.csv

🎨 Generating charts ...
  📊 Chart saved → ./gha_results/chart_smells_ground_truth.png
  📊 Chart saved → ./gha_results/chart_stages_ground_truth.png
  📊 Chart saved → ./gha_results/chart_smells_llm_output.png
  📊 Chart saved → ./gha_results/chart_stages_llm_output.png

📊 SMELL TYPE COMPARISON
  Smell Type                               Ground Truth    LLM Output
  ------------------------------------------------------------------
  AllowFailure                                        0             2
  BroadPermissions                                   25            45
  ContinueOnError                                  